In [1]:
from os.path import commonprefix
import attr
import json
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
from argparse import ArgumentParser
from src.utils import load_df
import pandas as pd
from src.structs import LemmatizeRule, WordForm
import attr
import pandas as pd
from src.categories import (
    name2mapping_to_id,
    name2mapping_from_id,
    names_order,
    UNDEFINED
)

from src.structs import LemmatizeRule, WordForm
from src.utils import load_df
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder

In [2]:
data_path = Path("/mnt/data_storage/datasets/generative_lemmatization_datasets/csvs")

In [3]:
# в test берем dev сплит чтобы в замерах финального пайплайна на настоящем test сплите в других работах не было утечки
df_train = load_df(data_path / "train.csv", dedup=True, drop_hapax=True)
df_test = load_df(data_path / "dev.csv", dedup=True, drop_hapax=True)

df_train = df_train.drop(["subset", "filename"], axis=1)
df_test = df_test.drop(["subset", "filename"], axis=1)

In [4]:
df_train.info()

<class 'pandas.DataFrame'>
Index: 106775 entries, 0 to 2148717
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   form    106775 non-null  str  
 1   pos     106775 non-null  str  
 2   feats   105689 non-null  str  
 3   lemma   106775 non-null  str  
dtypes: str(4)
memory usage: 4.1 MB


In [5]:
df_test.info()

<class 'pandas.DataFrame'>
Index: 18327 entries, 2 to 255882
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   form    18327 non-null  str  
 1   pos     18327 non-null  str  
 2   feats   17778 non-null  str  
 3   lemma   18327 non-null  str  
dtypes: str(4)
memory usage: 715.9 KB


In [6]:
df_train["feats"] = df_train["feats"].fillna("No Feats")
df_test["feats"] = df_test["feats"].fillna("No Feats")

In [7]:
df_train.info()

<class 'pandas.DataFrame'>
Index: 106775 entries, 0 to 2148717
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   form    106775 non-null  str  
 1   pos     106775 non-null  str  
 2   feats   106775 non-null  str  
 3   lemma   106775 non-null  str  
dtypes: str(4)
memory usage: 4.1 MB


In [8]:
df_test.info()

<class 'pandas.DataFrame'>
Index: 18327 entries, 2 to 255882
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   form    18327 non-null  str  
 1   pos     18327 non-null  str  
 2   feats   18327 non-null  str  
 3   lemma   18327 non-null  str  
dtypes: str(4)
memory usage: 715.9 KB


In [9]:
# у теста и трейна есть пересечения (высокочастотные слова и пунктуация). 
# поэтому test дополнитлеьно разделим на то что есть в train (known) и то чего нет (unknown)

cols = ['form', 'pos', 'feats', 'lemma']

key_train = df_train[cols].astype(str).agg("|".join, axis=1)
key_test = df_test[cols].astype(str).agg("|".join, axis=1)

mask = key_test.isin(key_train)

df_test_known = df_test[mask]
df_test_unkn = df_test[~mask]

In [10]:
print(f"Total test size: {df_test.shape[0]}; known size: {df_test_known.shape[0]}; unknown size: {df_test_unkn.shape[0]}")
assert df_test_known.shape[0] + df_test_unkn.shape[0] == df_test.shape[0]

Total test size: 18327; known size: 13500; unknown size: 4827


In [11]:
df_test_known.head()

,form,pos,feats,lemma
2,!,PUNCT,No Feats,!
5,Мы,PRON,Case:Nom Number:Plur Person:1 PronType:Prs,мы
6,Алиса,PROPN,Animacy:Anim Case:Nom Gender:Fem NameType:Giv ...,Алиса
7,и,CCONJ,No Feats,и
8,Коля,PROPN,Animacy:Anim Case:Nom Gender:Masc NameType:Giv...,Коля


In [12]:
df_test_unkn.head()

,form,pos,feats,lemma
3,Ставь,VERB,Aspect:Imp Mood:Imp Number:Sing Person:2 Trans...,ставить
4,❤️,SYM,No Feats,❤️
35,собираем,VERB,Aspect:Imp Mood:Ind Number:Plur Person:1 Tense...,собирать
51,подкинуть,VERB,Aspect:Perf Transit:Tran VerbForm:Inf Voice:Act,подкинуть
112,веселые,ADJ,Animacy:Inan Case:Acc Degree:Pos Number:Plur,веселый


In [13]:
## таким образом в known test split попала частотная лексика, которая естественным образом появляется и в трейне (из-за св-в естественного языка и речи)
## а в unknown test split попадают менее частотные случаи

In [14]:
## TODO: предобработка архаики? замена ятей, удаление терминалных твердых знаков и тд
## see https://github.com/ingoboerner/russpelling
## see https://sites.google.com/site/seesscm/pre-reform-russian-orthography-cheat-sheet?pli=1&authuser=0

In [15]:
# from https://github.com/DanAnastasyev/GramEval2020/blob/master/solution/train/lemmatize_helper.py
def predict_lemmatize_rule(word: str, lemma: str):
    def _predict_lemmatize_rule(word: str, lemma: str, **kwargs):
        if len(word) == 0:
            return LemmatizeRule(append_suffix=lemma, **kwargs)

        common_prefix = commonprefix([word, lemma])
        if len(common_prefix) == 0:
            rule = _predict_lemmatize_rule(word[1:], lemma, **kwargs)
            return attr.evolve(rule, cut_prefix=rule.cut_prefix + 1)

        return LemmatizeRule(cut_suffix=len(word) - len(common_prefix),
                                append_suffix=lemma[len(common_prefix):], **kwargs)

    word, lemma = word.replace('ё', 'е'), lemma.replace('ё', 'е')
    return min([
        _predict_lemmatize_rule(word, lemma),
        _predict_lemmatize_rule(word.lower(), lemma, lower=True),
        _predict_lemmatize_rule(word.capitalize(), lemma, capitalize=True),
        _predict_lemmatize_rule(word.upper(), lemma, upper=True)
    ], key=lambda rule: rule.cut_prefix + rule.cut_suffix)

In [16]:
# предскажем правила лемматизации для всех слов, но работать будем только с правилами из трейна

tqdm.pandas(desc="Подготовка правил лемматизации")

for df in (df_train, df_test, df_test_known, df_test_unkn):
    def _foo(row):
        return predict_lemmatize_rule(row["form"], row["lemma"])
    df["lemmatization_rule"] = df.progress_apply(
        _foo,
        axis=1
    )

Подготовка правил лемматизации: 100%|██████████| 4827/4827 [00:00<00:00, 35737.13it/s]


In [18]:
unique_rules = set(df_train["lemmatization_rule"].unique())
len(unique_rules)

1816

In [19]:
def _sort_rules(rules):
    return sorted(
        rules, key=lambda r: (
            r.cut_prefix, r.cut_suffix, r.append_suffix, r.lower, r.capitalize, r.upper
        )
    )

In [20]:
id2rule: dict[int, LemmatizeRule]= {
        i: rule for i, rule
        in enumerate(_sort_rules(unique_rules))
    }
rule2id = {rule: i for i, rule in id2rule.items()}

In [21]:
for df in (df_train, df_test, df_test_known, df_test_unkn):
    df["rule_id"] = df["lemmatization_rule"].map(rule2id)

In [22]:
# почти для всех случаев правила определяются в трейне!
# неопределенные случаи надо заполнить и заценить, что там
df_test_unkn.info()

<class 'pandas.DataFrame'>
Index: 4827 entries, 3 to 254911
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   form                4827 non-null   str    
 1   pos                 4827 non-null   str    
 2   feats               4827 non-null   str    
 3   lemma               4827 non-null   str    
 4   lemmatization_rule  4827 non-null   object 
 5   rule_id             4746 non-null   float64
dtypes: float64(1), object(1), str(4)
memory usage: 264.0+ KB


In [23]:
UNDEFINED_RULE_ID = -100

In [24]:
df_test_unkn["rule_id"] = df_test_unkn["rule_id"].fillna(UNDEFINED_RULE_ID)

In [25]:
unkn_test_undefined_rule = df_test_unkn[df_test_unkn["rule_id"] == UNDEFINED_RULE_ID]

In [26]:
unkn_test_undefined_rule.shape[0]

81

In [27]:
unkn_test_undefined_rule.head(15)

,form,pos,feats,lemma,lemmatization_rule,rule_id
3552,подольше,ADV,Degree:Cmp2,долго,"LemmatizeRule(cut_prefix=2, cut_suffix=3, appe...",-100.0
6465,гложет,VERB,Aspect:Imp Mood:Ind Number:Sing Person:3 Tense...,глодать,"LemmatizeRule(cut_prefix=0, cut_suffix=3, appe...",-100.0
10097,Апрѣля,NOUN,Animacy:Inan Case:Gen Gender:Masc Number:Sing,апрель,"LemmatizeRule(cut_prefix=0, cut_suffix=3, appe...",-100.0
10394,вѣрностію,NOUN,Animacy:Inan Case:Ins Gender:Fem Number:Sing,верность,"LemmatizeRule(cut_prefix=0, cut_suffix=8, appe...",-100.0
10635,изстари,ADV,Degree:Pos,исстари,"LemmatizeRule(cut_prefix=0, cut_suffix=6, appe...",-100.0
10702,помѣстья,NOUN,Animacy:Inan Case:Acc Gender:Neut Number:Plur,поместье,"LemmatizeRule(cut_prefix=0, cut_suffix=5, appe...",-100.0
11451,вѣрноподданнымъ,ADJ,Case:Dat Degree:Pos Number:Plur,верноподданный,"LemmatizeRule(cut_prefix=0, cut_suffix=14, app...",-100.0
12189,наслѣдственныхъ,ADJ,Case:Gen Degree:Pos Number:Plur,наследственный,"LemmatizeRule(cut_prefix=0, cut_suffix=11, app...",-100.0
12367,Намѣстничествѣ,NOUN,Animacy:Inan Case:Loc Gender:Neut Number:Sing,наместничество,"LemmatizeRule(cut_prefix=0, cut_suffix=11, app...",-100.0
12802,Генералъ-Губернатора,NOUN,Animacy:Anim Case:Gen Gender:Masc Number:Sing,генерал-губернатор,"LemmatizeRule(cut_prefix=0, cut_suffix=13, app...",-100.0


In [28]:
# доля случаев с неопределенным парвилом (правило не встретилось в тренировочных данных)
# часто это связвно с дореформенной орфографией. возможно есть смысл добавить правило на ее нормализацию, см выше.
unkn_test_undefined_rule.shape[0] / df_test_unkn.shape[0]

0.01678060907395898

In [29]:
## top k most frequent rules in train

k = 10
topk_rule_ids = df_train['rule_id'].value_counts().head(k).rename_axis('rule_id').reset_index(name='count')

for row in topk_rule_ids.itertuples(index=False):
    print(f"\n{'='*80}")
    print(f"rule_id: {row.rule_id}  |  count: {row.count}")
    print(f"rule: {id2rule[row.rule_id]}")
    subset = df_train[df_train['rule_id'] == row.rule_id]
    if set(subset["pos"]) == set(df_train["pos"]):
        pos_set = "ALL TAGS!!!"
    else:
        pos_set = sorted(subset['pos'].dropna().unique())
    print(f"pos tags: {pos_set}")
    print('-'*80)
    ex = subset.sample(
        n=min(5, row.count), random_state=42
    )
    for _, r in ex.iterrows():
        print(f"  form={r['form']!r:15}  lemma={r['lemma']!r:15}  pos={r['pos']!r:10}  feats={r['feats']}")


rule_id: 0  |  count: 23207
rule: LemmatizeRule(cut_prefix=0, cut_suffix=0, append_suffix='', lower=False, capitalize=False, upper=False)
pos tags: ALL TAGS!!!
--------------------------------------------------------------------------------
  form='запоминание'    lemma='запоминание'    pos='NOUN'      feats=Animacy:Inan Case:Acc Gender:Neut Number:Sing
  form='пристрастие'    lemma='пристрастие'    pos='NOUN'      feats=Animacy:Inan Case:Acc Gender:Neut Number:Sing
  form='Франсуа'        lemma='Франсуа'        pos='PROPN'     feats=Animacy:Anim Case:Nom Gender:Masc InflClass:Ind NameType:Giv Number:Sing
  form='равенство'      lemma='равенство'      pos='NOUN'      feats=Animacy:Inan Case:Acc Gender:Neut Number:Sing
  form='макрух'         lemma='макрух'         pos='X'         feats=Foreign:Yes

rule_id: 61  |  count: 9229
rule: LemmatizeRule(cut_prefix=0, cut_suffix=1, append_suffix='', lower=False, capitalize=False, upper=False)
pos tags: ['ADJ', 'ADP', 'ADV', 'ADVPRO', 'ANUM', '

In [30]:
# def get_pref(row):
#     return row["form"][:row["lemmatization_rule"].cut_prefix]

In [31]:
df_train = df_train.drop("lemmatization_rule", axis=1)

In [32]:
for n in range(1, 4):
    colname = f"suf{n}ch"
    df_train[colname] = df_train["form"].apply(lambda s: "<pad>"*max(n - len(s), 0) + s[-n:])


In [33]:
## гапаксы вроде убраны, а все равно матрицы будут огромные
for col in ("suf1ch", "suf2ch", "suf3ch",):
    print(col, df_train[col].nunique())

suf1ch 192
suf2ch 1412
suf3ch 6608


In [34]:
def parse_feats(feats: str, names_order=names_order, name2mapping_to_id=name2mapping_to_id) -> dict:

    result = {name: UNDEFINED for name in names_order}

    if not isinstance(feats, str) or feats == "No Feats":
        return result

    for part in filter(lambda elem: ":" in elem, feats.split()):
        k, v = part.split(":", 1)

        if k in result and v in name2mapping_to_id[k]:
            result[k] = v

    return result

feats_dicts = [parse_feats(feats) for feats in df_train["feats"].tolist()]
feats_df = pd.DataFrame(feats_dicts, index=df_train.index)
feats_df["upos"] = df_train["pos"]

df_train = pd.concat([df_train, feats_df], axis=1)

In [35]:
df_train.head()

,form,pos,feats,lemma,rule_id,suf1ch,suf2ch,suf3ch,upos,Animacy,...,NumType,Variant,PronType,Abbr,Poss,InflClass,Reflex,Polarity,Foreign,Hyph
0,Снова,ADV,Degree:Pos,снова,3,а,ва,ова,ADV,[UND],...,[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND]
1,приобрел,VERB,Aspect:Perf Gender:Masc Mood:Ind Number:Sing T...,приобрести,109,л,ел,рел,VERB,[UND],...,[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND]
2,дозу,NOUN,Animacy:Inan Case:Acc Gender:Fem Number:Sing,доза,64,у,зу,озу,NOUN,Inan,...,[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND]
3,",",PUNCT,No Feats,",",0,",","<pad>,","<pad><pad>,",PUNCT,[UND],...,[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND]
4,В,ADP,No Feats,в,3,В,<pad>В,<pad><pad>В,ADP,[UND],...,[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND]


In [36]:
df_train.columns

Index(['form', 'pos', 'feats', 'lemma', 'rule_id', 'suf1ch', 'suf2ch',
       'suf3ch', 'upos', 'Animacy', 'Case', 'Gender', 'Number', 'NameType',
       'Aspect', 'Mood', 'Tense', 'Transit', 'VerbForm', 'Voice', 'Degree',
       'Person', 'NumForm', 'NumType', 'Variant', 'PronType', 'Abbr', 'Poss',
       'InflClass', 'Reflex', 'Polarity', 'Foreign', 'Hyph'],
      dtype='str')

In [37]:
G_cols = ["suf1ch", "suf2ch", "suf3ch"]
M_cols = ['upos', 'Animacy', 'Case', 'Gender', 'Number', 'NameType',
       'Aspect', 'Mood', 'Tense', 'Transit', 'VerbForm', 'Voice', 'Degree',
       'Person', 'NumForm', 'NumType', 'Variant', 'PronType', 'Abbr', 'Poss',
       'InflClass', 'Reflex', 'Polarity', 'Foreign', 'Hyph']
tgt_col = ["rule_id"]

In [38]:
enc = OneHotEncoder(handle_unknown='error', sparse_output=False)

In [39]:
# будем дропать дубликаты, важно дропнуть и по таргетному правилу тоже
# теперь работаем с уникальными морфологическими разборами, которые мапятся на разные правила.
# таким образом можно попробовать найти те из них, которые детерминированно мапятся на одно и то же правило
df_train_M = df_train[M_cols + tgt_col].drop_duplicates()
df_train_M.shape

(8506, 26)

In [70]:
deterministic_M = []

for name, group in df_train_M.groupby("rule_id"):
    if group.shape[0] == 1:
        deterministic_M.append(group)

In [71]:
len(deterministic_M)

915

In [72]:
deterministic_M[0]

,upos,Animacy,Case,Gender,Number,NameType,Aspect,Mood,Tense,Transit,...,Variant,PronType,Abbr,Poss,InflClass,Reflex,Polarity,Foreign,Hyph,rule_id
8697,ADP,[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],...,[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],4


In [73]:
id2rule[4]

LemmatizeRule(cut_prefix=0, cut_suffix=0, append_suffix='-за', lower=False, capitalize=False, upper=False)

In [74]:
df_train[df_train["rule_id"] == 4]

,form,pos,feats,lemma,rule_id,suf1ch,suf2ch,suf3ch,upos,Animacy,...,NumType,Variant,PronType,Abbr,Poss,InflClass,Reflex,Polarity,Foreign,Hyph
8697,из,ADP,No Feats,из-за,4,з,из,<pad>из,ADP,[UND],...,[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND],[UND]


In [41]:
X = enc.fit_transform(df_train_M[M_cols])

In [42]:
y = df_train_M[tgt_col]

In [51]:
y.nunique()

rule_id    1816
dtype: int64

In [62]:
tree = DecisionTreeClassifier(
    criterion="gini",
    splitter="best",
    max_depth=10,     
    min_samples_leaf=20,
    min_samples_split=2,
    max_features=None,
    class_weight=None,
    random_state=42,
)

In [63]:
tree.fit(X, y)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",20
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples 

In [64]:
preds = tree.apply(X)

In [65]:
from sklearn.metrics import accuracy_score, f1_score

In [66]:
accuracy_score(y, preds)

0.00047025628967787447

In [67]:
f1_score(y, preds, average="weighted")

8.25350204627722e-05

In [68]:
from sklearn.tree import export_text

In [69]:
print(export_text(tree, feature_names=list(enc.get_feature_names_out(M_cols))))

|--- InflClass_Ind <= 0.50
|   |--- VerbForm_Part <= 0.50
|   |   |--- VerbForm_[UND] <= 0.50
|   |   |   |--- Voice_Mid <= 0.50
|   |   |   |   |--- Tense_Past <= 0.50
|   |   |   |   |   |--- Aspect_[UND] <= 0.50
|   |   |   |   |   |   |--- Person_3 <= 0.50
|   |   |   |   |   |   |   |--- Number_Plur <= 0.50
|   |   |   |   |   |   |   |   |--- VerbForm_Inf <= 0.50
|   |   |   |   |   |   |   |   |   |--- Mood_Ind <= 0.50
|   |   |   |   |   |   |   |   |   |   |--- class: 117
|   |   |   |   |   |   |   |   |   |--- Mood_Ind >  0.50
|   |   |   |   |   |   |   |   |   |   |--- class: 351
|   |   |   |   |   |   |   |   |--- VerbForm_Inf >  0.50
|   |   |   |   |   |   |   |   |   |--- Transit_Tran <= 0.50
|   |   |   |   |   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |   |   |   |   |--- Transit_Tran >  0.50
|   |   |   |   |   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |   |   |--- Number_Plur >  0.50
|   |   |   |   |   |   |   |   |--- Person_2 <= 0.50
|  

In [81]:
def check_InflClass_Ind_rule(df):
    return df[(df["feats"].str.contains("InflClass:Ind")) & (df["rule_id"] != 0) & (df["form"].str.lower() != df["lemma"].str.lower())]

In [82]:
check_InflClass_Ind_rule(df_train)

,form,pos,feats,lemma,rule_id,suf1ch,suf2ch,suf3ch,upos,Animacy,...,NumType,Variant,PronType,Abbr,Poss,InflClass,Reflex,Polarity,Foreign,Hyph
178421,Джефри,PROPN,Animacy:Anim Case:Nom Gender:Masc InflClass:In...,Джеффри,357,и,ри,фри,PROPN,Anim,...,[UND],[UND],[UND],[UND],[UND],Ind,[UND],[UND],[UND],[UND]
668708,Мерфи,PROPN,Animacy:Anim Case:Gen Gender:Masc InflClass:In...,Мэрфи,920,и,фи,рфи,PROPN,Anim,...,[UND],[UND],[UND],[UND],[UND],Ind,[UND],[UND],[UND],[UND]
1537394,Пинокио,PROPN,Animacy:Anim Case:Gen Gender:Masc InflClass:In...,Пиноккио,253,о,ио,кио,PROPN,Anim,...,[UND],[UND],[UND],[UND],[UND],Ind,[UND],[UND],[UND],[UND]
1537550,Пинокио,PROPN,Animacy:Anim Case:Loc Gender:Masc InflClass:In...,Пиноккио,253,о,ио,кио,PROPN,Anim,...,[UND],[UND],[UND],[UND],[UND],Ind,[UND],[UND],[UND],[UND]
1578212,тысячу,NOUN,Animacy:Inan Case:Acc Gender:Fem InflClass:Ind...,тысяча,64,у,чу,ячу,NOUN,Inan,...,[UND],[UND],[UND],[UND],[UND],Ind,[UND],[UND],[UND],[UND]


In [83]:
check_InflClass_Ind_rule(df_test)

,form,pos,feats,lemma,lemmatization_rule,rule_id
